# Demo 04 : Loop detection

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · watchdog-node

We let a model propose another inspection while a watchdog measures progress in state. A scripted lack of progress makes the stopping condition repeatable. The recursion limit remains an outer bound on graph steps.

In [2]:
import json
import operator
import sqlite3
import time
from pathlib import Path
from typing import Annotated, Literal, TypedDict
from uuid import uuid4

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Usage(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.calls += 1

    def on_llm_end(self, response, **kwargs):
        for batch in response.generations:
            for generation in batch:
                usage = generation.message.usage_metadata or {}
                self.input_tokens += usage.get("input_tokens", 0)
                self.output_tokens += usage.get("output_tokens", 0)

usage = Usage()
llm = make_llm(callbacks=[usage], max_retries=0, max_tokens=650)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as conn:
    conn.row_factory = sqlite3.Row
    work_order = dict(conn.execute("SELECT * FROM work_orders WHERE wo_id = ?", ("BW-7001",)).fetchone())
    part = dict(conn.execute("SELECT * FROM parts WHERE part_no = ?", ("CE-CF-02",)).fetchone())
print("work order:", work_order)
print("part:", part)

def config(label):
    return {"configurable": {"thread_id": f"07-{label}-{uuid4()}"}, "recursion_limit": 30}

work order: {'wo_id': 'BW-7001', 'fleet_no': 'CE12-2021-0117', 'opened': '2026-09-14', 'status': 'open', 'error_code': 'F-CE-110', 'summary': 'Traction system over-temperature during afternoon route', 'mechanic_id': 'M01'}
part: {'part_no': 'CE-CF-02', 'model': 'CE-12', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}


In [3]:
from langgraph.errors import GraphRecursionError

class Investigation(TypedDict):
    attempts: int
    evidence: str
    previous: str
    stalled: int
    proposal: str
    status: str

def investigate(state):
    proposal = llm.invoke(f"Suggest a short inspection action for {work_order}. Evidence remains unchanged: {state['evidence']}. Do not claim an inspection was performed.").content
    return {"attempts": state["attempts"] + 1, "proposal": proposal}

def watchdog(state):
    stalled = state["stalled"] + 1 if state["evidence"] == state["previous"] else 0
    stop = stalled >= 2 or state["attempts"] >= 4
    print("watchdog:", state["attempts"], stalled, "stop" if stop else "continue")
    return {"previous": state["evidence"], "stalled": stalled,
            "status": "manual review" if stop else "continue"}

def route(state) -> Literal["investigate", "manual"]:
    return "manual" if state["status"] == "manual review" else "investigate"

watch = StateGraph(Investigation)
watch.add_node("investigate", investigate)
watch.add_node("watchdog", watchdog)
watch.add_node("manual", lambda s: {"status": "manual review queued"})
watch.add_edge(START, "investigate")
watch.add_edge("investigate", "watchdog")
watch.add_conditional_edges("watchdog", route)
watch.add_edge("manual", END)
initial = {"attempts": 0, "evidence": work_order["summary"], "previous": work_order["summary"], "stalled": 0, "proposal": "", "status": ""}
out = watch.compile().invoke(initial, {"recursion_limit": 12})
print("status:", out["status"])
print("proposal:", out["proposal"])
assert out["attempts"] == 2 and out["status"] == "manual review queued"

bound = StateGraph(Investigation)
bound.add_node("spin", lambda s: {"attempts": s["attempts"] + 1})
bound.add_edge(START, "spin")
bound.add_edge("spin", "spin")
try:
    bound.compile().invoke(initial, {"recursion_limit": 4})
except GraphRecursionError as exc:
    print("outer bound:", type(exc).__name__)

watchdog: 1 1 continue


watchdog: 2 2 stop
status: manual review queued
proposal: Inspect the traction cooling system and check for blocked airflow, coolant leaks, or abnormal temperatures.
outer bound: GraphRecursionError


## b · repeated-tool-calls

We obtain a stock tool call from the course model, then replay the same name and arguments with a new call id. We compare canonical arguments before execution. Identical read requests without new evidence stop the run; call limits from module 04 provide a separate bound.

In [4]:
@tool
def read_stock(part_no: str) -> str:
    """Read stock for the supported Veenhaven Bus part."""
    return json.dumps(part) if part_no == part["part_no"] else "Unknown part"

proposed = llm.bind_tools([read_stock], tool_choice="read_stock", parallel_tool_calls=False).invoke(
    f"Read stock for {part['part_no']}."
)
assert len(proposed.tool_calls) == 1
first = proposed.tool_calls[0]
repeated = AIMessage("", tool_calls=[{**first, "id": "repeat-2"}])

class Calls(MessagesState):
    seen: list[str]
    executed: int
    blocked: bool

def signature(call):
    return json.dumps({"name": call["name"], "args": call["args"]}, sort_keys=True)

def guard(state):
    calls = state["messages"][-1].tool_calls
    duplicate = any(signature(call) in state["seen"] for call in calls)
    return {"blocked": duplicate}

def execute(state):
    call = state["messages"][-1].tool_calls[0]
    return {"messages": [ToolMessage(read_stock.invoke(call["args"]), tool_call_id=call["id"])],
            "seen": [*state["seen"], signature(call)], "executed": state["executed"] + 1}

def replay(state):
    return {"messages": [repeated]}

def stop(state):
    call = state["messages"][-1].tool_calls[0]
    return {"messages": [ToolMessage("Repeated unchanged read blocked; manual review required.", tool_call_id=call["id"], status="error")]}

calls = StateGraph(Calls)
calls.add_node("guard", guard)
calls.add_node("execute", execute)
calls.add_node("replay", replay)
calls.add_node("stop", stop)
calls.add_edge(START, "guard")
calls.add_conditional_edges("guard", lambda s: "stop" if s["blocked"] else "execute", {"stop": "stop", "execute": "execute"})
calls.add_edge("execute", "replay")
calls.add_edge("replay", "guard")
calls.add_edge("stop", END)
out = calls.compile().invoke({"messages": [HumanMessage("Check stock"), proposed], "seen": [], "executed": 0, "blocked": False})
print("signature:", out["seen"])
print("executed:", out["executed"], "blocked:", out["blocked"])
print("result:", out["messages"][-1].content)
assert out["executed"] == 1 and out["blocked"]

signature: ['{"args": {"part_no": "CE-CF-02"}, "name": "read_stock"}']
executed: 1 blocked: True
result: Repeated unchanged read blocked; manual review required.


We scope duplicate detection to a run with unchanged evidence. Polling live data can legitimately repeat a read; a policy can include an evidence version or a freshness window. A successful tool call and its retry are separate from an agent asking again.

In [5]:
print("metrics:", json.dumps({"model": MODEL, "calls": usage.calls,
      "input_tokens": usage.input_tokens, "output_tokens": usage.output_tokens,
      "seconds": round(time.perf_counter() - started, 1)}))

metrics: {"model": "openai/gpt-6-luna", "calls": 3, "input_tokens": 351, "output_tokens": 71, "seconds": 9.3}


## Things to try

Optional : we can change the replayed arguments or reset stalled progress when new inspection evidence arrives.